# UniAudio — Python quickstart

`uniaudio` is a Cython extension over the UniAudio C ABI. It reads audio
containers, decodes the ones under no licence, reads tags, and fingerprints
a recording by how it sounds.

It is a thin binding: what the C ABI cannot reach, this cannot reach either.

```
pip install lituus-uniaudio
```

The cells below read the repository's own test fixtures, so this notebook
runs from where it sits in the source tree.

In [1]:
import importlib.resources

import uniaudio

# The audio below ships inside the wheel, so this notebook runs wherever
# uniaudio is installed -- not only inside a checkout.
FIXTURES = importlib.resources.files("uniaudio") / "data"
uniaudio.version()

'0.1.0'

## Naming a container without decoding it

The extension is not evidence of what a file holds. `sniff` reads the
leading bytes and says what is really there.

In [2]:
for name in ["sweep.wav", "sweep.flac", "sweep-alac.m4a",
             "sweep-vorbis.ogg", "sweep-mp3.mp3"]:
    print(f"{name:<18} {uniaudio.sniff(FIXTURES / name)}")

sweep.wav          wav
sweep.flac         flac
sweep-alac.m4a     mp4
sweep-vorbis.ogg   ogg
sweep-mp3.mp3      mp3


## The shape of the audio

`probe` returns `(sample_rate, channels, frames)` for any container the
library decodes. `frames` counts per channel.

These five files are the same three seconds of a sine sweep.

In [3]:
for name in ["sweep.wav", "sweep.flac", "sweep-alac.m4a",
             "sweep-vorbis.ogg", "sweep-mp3.mp3"]:
    rate, channels, frames = uniaudio.probe(FIXTURES / name)
    print(f"{name:<18} {rate} Hz, {channels} ch, {frames} frames")

sweep.wav          11025 Hz, 1 ch, 33075 frames
sweep.flac         11025 Hz, 1 ch, 33075 frames
sweep-alac.m4a     11025 Hz, 1 ch, 33075 frames
sweep-vorbis.ogg   11025 Hz, 1 ch, 33075 frames
sweep-mp3.mp3      11025 Hz, 1 ch, 33075 frames


## Tags

Four unrelated tagging schemes grew up around these formats — ID3 in MPEG
audio, Vorbis comments in Ogg and FLAC, iTunes atoms in MP4. `tags` reads
whichever one a file uses into the same dictionary.

In [4]:
uniaudio.tags(FIXTURES / "tagged-v24.mp3")

{'title': 'Été à Nice',
 'artist': 'Lituus Lab',
 'album': 'Fixtures',
 'albumArtist': '',
 'composer': '',
 'genre': 'Ambient',
 'comment': 'one line',
 'date': '2026',
 'trackNumber': 3,
 'trackTotal': 12,
 'discNumber': 0,
 'discTotal': 0,
 'other': [{'key': 'TSSE', 'value': 'Lavf63.1.101'}]}

`date` is whatever the file wrote, kept as a string. Tag dates follow no
agreed format, and deciding which half of a bare `01/02/2019` is the month
would be an invention.

A name with no field of its own is not dropped; it lands in `other`.

## Recognising a recording by how it sounds

The fingerprint describes the sound, not the bytes, so two encodings of the
same audio give nearly the same one. It finds duplicates that no checksum
would match.

In [5]:
duration, words = uniaudio.fingerprint(FIXTURES / "sweep.wav")
print(f"{duration:.2f} s, {len(words)} words")
print(words[:6])

3.00 s, 21 words
[511590015, 4194138623, 4093110269, 3485978617, 2680143828, 2102361881]


In [6]:
_, from_flac = uniaudio.fingerprint(FIXTURES / "sweep.flac")
_, from_mp3 = uniaudio.fingerprint(FIXTURES / "sweep-mp3.mp3")

print("wav against flac:", uniaudio.similarity(words, from_flac))
print("wav against mp3: ", uniaudio.similarity(words, from_mp3))

wav against flac: 1.0
wav against mp3:  0.7738095238095238


FLAC is lossless, so its fingerprint matches exactly. A lossy encode moves
it, and a pure sweep is the hardest case there is: nearly all its energy
sits in one band at a time, so a small change flips many bits at once. Do
not read the second number as what the fingerprint does to music.

## When a file cannot be read

Every failure is a `UniAudioError` carrying both the reason the library gave
and a status: `2` for a file that is not there, `3` for bytes that are not
what they claim. The two are different problems and are worth telling apart.

In [7]:
try:
    uniaudio.probe("no-such-file.flac")
except uniaudio.UniAudioError as failure:
    print(f"status {failure.status}: {failure}")

status 2: cannot open no-such-file.flac


A container the library reads but a codec it does not decode is the second
kind. The file below is a real Ogg — the container parses — carrying FLAC
rather than Vorbis, and the error says so instead of guessing.

In [8]:
try:
    uniaudio.probe(FIXTURES / "sweep-oggflac.ogg")
except uniaudio.UniAudioError as failure:
    print(f"status {failure.status}: {failure}")

status 3: ogg: the stream is flac, not vorbis
